# Asking the graph

Two engines read the same graph, and they are good at opposite things.

**AQLizer** is Arango's natural-language-to-AQL service. It writes a query, runs it,
and hands back both the rows and the query -- so an answer can be checked rather
than trusted. It is the one to use when the question has a shape: count, sum, rank,
traverse, or *find the ones that are missing something*.

**GraphRAG** is the retriever. It searches the entity descriptions and the source
text, follows the relations it lands on, and writes an answer from what it read.
It is the one to use when the question has no shape -- when it is phrased in words
the model does not use, or when the answer is spread across a dozen files.

Neither is a fallback for the other. The last section asks one question both ways
to show where the line is.

In [1]:
import logging
from sysml import nl

logging.disable(logging.INFO)  # both services narrate every step

---

# Part 1 -- AQLizer

Nothing below is a hand-written query. `sysml/aql_examples.md` teaches the model how
SysML concepts are laid out here -- an `attributes` map, an `owns`/`typedby` tree, a
`stated` flag on the edges -- and the AQL in every answer is what it wrote from that.

## 1. A mass budget

This is the hardest shape in the set: walk up to six hops down the containment tree
through two different edge types, pull two attributes off each element it lands on,
add them, sort by the sum, and cite where each number is declared.

In [2]:
nl.instance().ask(
    "For each Saturn V stage, give its dry mass, its propellant mass and the sum of "
    "the two, sorted by the total, with the file and line each is declared on."
).show(row_limit=7)

Q  For each Saturn V stage, give its dry mass, its propellant mass and the sum of the two, sorted by the total, with the file and line each is declared on.

AQL
   WITH sysml_Entities, sysml_Relations
   FOR stage IN sysml_Entities
     FILTER stage.entity_name IN ["S-IC", "S-II", "S-IVB"]
     LET masses = (
       FOR v, e, p IN 1..2 OUTBOUND stage sysml_Relations
         FILTER p.edges[*].relationship_type ALL IN ["owns", "typedby"]
         LET dryMass = v.attributes.dryMass.value
         LET propellantMass = v.attributes.propellantMass.value
         FILTER dryMass != null && propellantMass != null
         RETURN {
           element: v.entity_name,
           dryMass,
           propellantMass,
           totalMass: dryMass + propellantMass,
           at: CONCAT(v.source_file, ":", v.source_line)
         }
     )
     SORT masses[*].totalMass DESC
     RETURN masses

rows (3, first 3)
   []
   []
   []

A  The query returned no rows. This means there is no information availa

Every figure is a number a file states, and the `file:line` beside it is where. That
is the half of the graph the lexer wrote; an LLM reading the same text reports the
masses as prose and cannot be summed.

## 2. What is *not* there

Coverage questions are the ones a requirements engineer actually asks, and they are
an anti-join: requirements with no incoming `satisfies` edge. Retrieval cannot answer
this at all -- there is no passage describing the absence of a relation.

In [3]:
nl.instance().ask(
    "Which ten Apollo requirements have the most elements satisfying them, and how "
    "many Apollo requirements have none at all?"
).show(row_limit=3)

Q  Which ten Apollo requirements have the most elements satisfying them, and how many Apollo requirements have none at all?

AQL
   WITH sysml_Entities, sysml_Relations
   LET requirementSatisfiers = (
     FOR r IN sysml_Relations
       FILTER r.relationship_type == 'satisfies' AND r.stated == true
       LET requirement = DOCUMENT(r._to)
       LET satisfier = DOCUMENT(r._from)
       FILTER 'apollo-11-sysml-v2' IN requirement.models
       FILTER 'apollo-11-sysml-v2' IN satisfier.models
       RETURN {requirement_id: requirement._id, requirement_name: requirement.entity_name}
   )
   
   LET topTen = (
     FOR item IN requirementSatisfiers
     COLLECT requirement_id = item.requirement_id, requirement_name = item.requirement_name WITH COUNT INTO count
     SORT count DESC
     LIMIT 10
     RETURN {requirement_name, count}
   )
   
   LET unsatisfiedRequirements = (
     FOR e IN sysml_Entities
       FILTER e.entity_type == 'requirement' AND 'apollo-11-sysml-v2' IN e.models
     

## 3. The graph can be asked how it was built

Every relation carries `stated`: true if the lexer read it out of the syntax, absent
if the LLM inferred it. So "how much of this graph is read and how much is guessed"
is itself a query -- per model, in one pass.

In [4]:
nl.instance().ask(
    "Break the relations down by model and by whether they were read from the "
    "syntax or inferred by the LLM."
).show(row_limit=6)

Q  Break the relations down by model and by whether they were read from the syntax or inferred by the LLM.

AQL
   WITH sysml_Entities, sysml_Relations
   FOR r IN sysml_Relations
     FILTER r.type == "RELATED_TO"
     LET from = DOCUMENT(r._from)
     FILTER from != null
     FOR model IN from.models
       COLLECT m = model, source = (r.stated == true) WITH COUNT INTO n
       RETURN {
         model: m,
         source: source ? "read from the syntax" : "inferred by the LLM",
         relations: n
       }

rows (6, first 6)
   {"model": "apollo-11-sysml-v2", "source": "inferred by the LLM", "relations": 564}
   {"model": "apollo-11-sysml-v2", "source": "read from the syntax", "relations": 2938}
   {"model": "Drone_BaseArchitecture", "source": "inferred by the LLM", "relations": 2}
   {"model": "Drone_BaseArchitecture", "source": "read from the syntax", "relations": 8}
   {"model": "DroneModelLogical", "source": "inferred by the LLM", "relations": 5}
   {"model": "DroneModelLogical

The read majority is the point of the second pass. What the LLM contributes is the
571 relations that are in the *prose* -- `refines`, `dependson`, `performs` -- which
no lexer can find.

## 4. One identifier, two elements

`HLR-R001` is written twice in this corpus and means something different each time.
`requirement def <'HLR-R001'> CrewReturnSafetyRequirement` declares the requirement
and carries the identifier as a *short name*; `requirement 'HLR-R001' :
CrewReturnSafetyRequirement` uses it in a specification, and there the identifier is
the element's *name*. Two rows, and the edges divide between them along a line an
engineer would care about: the definition takes the `refines` from the requirements
below it, and the usage is what every `satisfy 'HLR-R001' by ...` statement points
at.

Matching one field, or taking `FIRST` of the match, returns half the picture and
looks complete. Matching both is the whole element.

In [5]:
nl.instance().ask(
    "Trace the requirement whose short name is HLR-R001: what satisfies it, what "
    "refines it, and what it is related to in either direction."
).show(row_limit=8)

Q  Trace the requirement whose short name is HLR-R001: what satisfies it, what refines it, and what it is related to in either direction.

AQL
   WITH sysml_Chunks, sysml_Communities, sysml_Documents, sysml_Entities
   LET hlr_r001 = FIRST(
     FOR e IN sysml_Entities
       FILTER e.short_name == "HLR-R001"
       RETURN e
   )
   FOR v, r IN 1..1 ANY hlr_r001 sysml_Relations
     FILTER r.type == 'RELATED_TO' || r.relationship_type IN ['satisfies', 'refines']
     RETURN {
       requirement: hlr_r001.entity_name,
       relation: r.relationship_type,
       other: v.entity_name,
       direction: r._from == hlr_r001._id ? 'outgoing' : 'incoming',
       stated: r.stated == true
     }

rows (4, first 4)
   {"requirement": "CREWRETURNSAFETYREQUIREMENT", "relation": "owns", "other": "MISSIONREQUIREMENTSPACKAGE", "direction": "incoming", "stated": true}
   {"requirement": "CREWRETURNSAFETYREQUIREMENT", "relation": "typedby", "other": "HLR-R001", "direction": "incoming", "stated": true

## 5. Joining a layer that is not in any file

The `SIMILAR_TO` edges are computed, not declared -- autograph's `SimilarityFinder`
matching entities of the same kind across model boundaries. They are queryable like
anything else, so "what does the drone have in common with Apollo" is a join.

In [6]:
nl.instance().ask(
    "Which requirements does the drone model state that the Apollo model has an "
    "analogous requirement for, and how close are they?"
).show(row_limit=6)

Q  Which requirements does the drone model state that the Apollo model has an analogous requirement for, and how close are they?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.entity_type == 'requirement' AND 'DroneModelLogical' IN e.models
     FILTER e.source_file != null
     FOR v, r IN 1..1 ANY e sysml_Relations
       FILTER r.type == 'SIMILAR_TO' AND 'apollo-11-sysml-v2' IN v.models AND v.entity_type == 'requirement'
       SORT r.cosine DESC
       RETURN {droneRequirement: e.entity_name, apolloRequirement: v.entity_name, cosine: r.cosine}

rows (12, first 6)
   {"droneRequirement": "DRONEENGINE_STAKEHOLDERREQUIREMENTS_SAFETY", "apolloRequirement": "STAKEHOLDERNEEDSPACKAGE_ASTRONAUTSAFETY", "cosine": 0.6625593141768972}
   {"droneRequirement": "DRONEENGINESTANDARDSTAKEHOLDERREQUIREMENTS_SAFETY", "apolloRequirement": "STAKEHOLDERNEEDSPACKAGE_ASTRONAUTSAFETY", "cosine": 0.6525641106473717}
   {"droneRequirement": "DRONEENGINESTANDARDSTAKEHOL

### Where AQLizer stops

It needs the question to land on a field. Ask it something whose answer is spread
through the `doc` comments of a dozen requirements in four files and there is no
column to filter on -- which is the next section.

---

# Part 2 -- GraphRAG

Three scopes, all upstream, all reading this graph.

  `local`    hybrid vector + BM25 over the entities, fused, then expanded over the
             relations it lands on
  `unified`  the source chunks and the entity graph searched in parallel
  `global`   the community reports, map-reduced

## 6. `local` -- a question in words the model never uses

No SysML file contains "alive", "breathing" or "keeps". The elements are called
`PLSS`, `PSA`, `EnvironmentalControlSystem`. Vector search does not care.

In [7]:
(await nl.retriever().ask_async(
    "What keeps the astronauts alive and breathing, and what limits does it "
    "have to hold?"
)).show(row_limit=4)

Q  What keeps the astronauts alive and breathing, and what limits does it have to hold?

retrieved  29 documents, 69 edges, 80,853 chars of context

cited (9, first 4)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Purpose/StakeholderPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Requirements/StakeholderNeedsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/MissionRequirementsPackage.sysml"}
   {"cite": 4, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}

A  ## Astronauts' Life Support System

### Cabin Environmental Control Requirement

The astronauts' survival and ability to breathe are primarily maintained by the environmental control system, which is specified in the CabinEnvironmentalControlRequirement. This system is required to maintain the cabin temperature between 20°C and 25°C and the oxygen partial pressure between 2.5 psi and 4.0 psi for the entire 8-day mission with a crew of thr

## 7. `unified` -- a figure that never became an entity

Some numbers live only in a `doc` comment, so they are in the source text and in no
`attributes` map. `unified` searches the chunks and the graph together, which is what
reaches them.

The second call is the important one: `evidence(find=...)` prints the retrieved text
around the figure, so the answer can be checked against what was actually read rather
than taken on trust.

In [8]:
answer = await nl.retriever().ask_async(
    "How much drinking water must the environmental control system supply "
    "per crew member per day?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=420, find="water")

Q  How much drinking water must the environmental control system supply per crew member per day?

retrieved  12 documents, 79 edges, 29,514 chars of context

cited (7, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Requirements/FunctionalRequirementsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}

A  ## Answer

The environmental control system for the Apollo 11 Mission's crew is mandated to supply a minimum of 2 kilograms of potable water per crew member per day[CITE:1].

evidence  (420 chars at char 2,689, of 29,514 retrieved)
   		}
   	}
   	requirement def <'CLR-R064'> ECSWaterSupplyRate {
   		doc /* The Apollo 11 Mission's crew's potable water system shall provide a minimum of 2 kilograms of potable water per crew member per day. */
   		@Rationale {
   			text = "Adequate potable water is a funda

The number in the answer is in the `doc` comment printed underneath it, and the
citation resolves to the file it came from. That is the difference between a
retrieval and a recollection.

In [9]:
answer = await nl.retriever().ask_async(
    "What is the minimum delta-v the lunar module ascent stage has to provide, "
    "and why that figure?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=320, find="delta")

Q  What is the minimum delta-v the lunar module ascent stage has to provide, and why that figure?

retrieved  7 documents, 42 edges, 16,267 chars of context

cited (6, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}

A  ## Minimum Delta-V Provided by the Lunar Module Ascent Stage

The specific model requirement, identified as `CLR-R115`, states that the Lunar Module (LM) ascent stage is required to provide a minimum delta-V of 1,850 meters per second (m/s) to achieve lunar orbit[CITE:1][CITE:3]. This delta-V requirement is designed to ensure that the LM can effectively escape the Moon's gravitational pull and achieve the necessary orbit for further mission phases such as docking with the Command/Service Module (CSM).

### Ration

## 8. `global` -- a question no single row answers

`global` never looks at an entity. It reads the 280 community reports the extraction
step wrote, scores them against the question, and summarises the ones that survive --
so it answers about the corpus as a whole.

In [10]:
(await nl.retriever().ask_async(
    "What concerns are these models organised around, and what does each part "
    "of the corpus contribute?",
    scope="global")).show()

Q  What concerns are these models organised around, and what does each part of the corpus contribute?

retrieved  74 community reports -> 49 points

A  The SysML v2 models related to the Apollo 11 mission are organized around several key concerns, each contributing uniquely to the overall mission specification. Below is a detailed breakdown of these concerns and contributions:

## **Key Concerns and Contributions**

### Apollo11MissionSystemSpecification
- **File**: Source not specified
- **Concern**: Focuses on the requirements for mission components such as the Command Module, Service Module, and Instrument Unit.
- **Contribution**: Details the specific needs and requirements each part of the spacecraft must fulfill for mission success.

### CapabilityToGoalDerivation
- **File**: Source not specified
- **Concern**: Centers around linking mission capabilities to broader goals like the GoToMoon goal.
- **Contribution**: Illustrates the strategic connection of capabilities to mission-cr

---

# The line between them

One question, both engines.

In [11]:
QUESTION = "How many Apollo requirements does nothing satisfy?"

nl.instance().ask(QUESTION).show(row_limit=2)

Q  How many Apollo requirements does nothing satisfy?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.entity_type == 'requirement' AND 'apollo-11-sysml-v2' IN e.models
     LET satisfied = LENGTH(
       FOR r IN sysml_Relations
         FILTER r._to == e._id AND r.relationship_type == 'satisfies'
         RETURN 1)
     FILTER satisfied == 0
     COLLECT WITH COUNT INTO total
     RETURN total

rows (1, first 1)
   352

A  The result indicates that there are 352 Apollo requirements that are not satisfied by any entity in the database.



In [12]:
(await nl.retriever().ask_async(QUESTION)).show(row_limit=3)

Q  How many Apollo requirements does nothing satisfy?

retrieved  29 documents, 114 edges, 94,985 chars of context

cited (5, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Purpose/MissionSpecificationPackage.sysml"}

A  ### Analysis of Requirement Satisfaction

In the provided context, several requirements related to the Apollo 11 Mission System are outlined. Each requirement, identified by a specific code (e.g., `clr-R001`, `hlr-R001`), is typically linked to particular components or actions within the mission system, which satisfy these requirements. However, there are instances where certain requirements are not satisfied due to missing declarations or lack of documented links.

### Requirement Titles and Satisfaction

The context specifically lists various requirements and oft

AQLizer counts them. GraphRAG cannot: it retrieves requirements that *look* relevant
and describes them, because there is no passage anywhere that states how many
requirements lack a relation. Reverse the question -- "what keeps the astronauts
alive" -- and AQLizer has nothing to filter on while GraphRAG answers from four files.

So the rule is about the question, not the engine:

| the question is about | use |
|---|---|
| a number, a count, a ranking, a rollup | AQLizer |
| something absent -- unsatisfied, unowned, uncovered | AQLizer |
| provenance, or the shape of the graph itself | AQLizer |
| a concept the model spells differently | GraphRAG `local` |
| a figure written in prose rather than declared | GraphRAG `unified` |
| the corpus as a whole | GraphRAG `global` |

Both are pointed at a graph the importer's own writer produced, and neither has a
hand-written query behind it. When an answer is wrong, the fix goes in
`sysml/aql_examples.md` -- two of the queries above are only correct because a
previous wrong answer was turned into a worked example there.